In [13]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
import sys

from torch.utils.data import DataLoader

sys.path.append('..')

from models.SRCNN.model import SRCNN
from models.SRCNN import config

from datasets.super_res_dataset_fourier import SuperResDataset
from datasets.sr_tiny_dataset import SRTinyDataset

In [10]:
test_dataset = SuperResDataset(
    hr_files=['../data/100/window_2023.npy'],
    downsample_factor=1,
    freq_threshold=5
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SRCNN(2,2)
model = torch.nn.DataParallel(model)
model = model.to(device)

checkpoint = torch.load(
    "../weights/SRCNN_best_freq.pth",
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

(1000, 128, 128, 6)


DataParallel(
  (module): SRCNN(
    (activation): ReLU()
    (conv1): Conv2d(2, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
    (conv2): Conv2d(64, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (conv3): Conv2d(32, 2, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  )
)

In [14]:
def evaluate_model(model, test_loader, device):
    model.eval()
    
    metrics = {
        'mse': {'u': [], 'v': []},
        'ssim': {'u': [], 'v': []},
        'psnr': {'u': [], 'v': []},
        'lpips': {'u': [], 'v': []},
        'fsim': {'u': [], 'v': []},
        'epi':{'u': [], 'v': []},
        'tke': {'u': [], 'v': []}
    }
    
    with torch.no_grad():
        for idx, (lr, hr) in tqdm(enumerate(test_loader)):
            if idx >= 10: break
            lr, hr = lr.to(device), hr.to(device)
            output = model(lr)
            # Compute metrics for each channel
            for channel in range(2):  # u and v components
                channel_label = 'u' if channel == 0 else 'v'
                metrics['mse'][f'{channel_label}'].append(mseLoss(output, hr, channel, idx))
                metrics['ssim'][f'{channel_label}'].append(ssimLoss(output, hr, channel, idx))
                metrics['psnr'][f'{channel_label}'].append(psnrLoss(output, hr, channel, idx))
                metrics['lpips'][f'{channel_label}'].append(lpipsLoss(output, hr, channel, idx))
                metrics['fsim'][f'{channel_label}'].append(fsimLoss(output, hr, channel, idx))
                metrics['epi'][f'{channel_label}'].append(epiLoss(output, hr, channel, idx))
                metrics['tke'][f'{channel_label}'].append(LossTKE(output, hr, channel, idx))

    # Aggregate results
    results = {k: {'u': {'mean': np.mean(v['u']), 'std': np.std(v['u'])},
                    'v': {'mean': np.mean(v['v']), 'std': np.std(v['v'])}}
               for k, v in metrics.items()}
    return results

In [15]:
results = evaluate_model(model, test_loader, device)
print(results)

0it [00:00, ?it/s]


NameError: name 'mseLoss' is not defined